# Problem 1. 신용카드 이상 거래 탐지하기

## 문제 개요
카드사는 승인 요청이 들어온 거래 중 **부정 사용(사기) 거래**를 실시간으로 찾아내야 합니다.
거래 정보로 각 거래가 사기일 **확률**을 예측하세요.

## 파일 구조
```
Problem1/
├── problem1.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 70,000건 (Class 포함, 사기 358건)
- `data/test.csv` : 30,000건 (Class 없음)

| 컬럼 | 설명 |
|---|---|
| id | 거래 고유 번호 |
| Time | 데이터 수집 시작 이후 경과 시간(초), 약 이틀치 |
| V1 ~ V28 | 개인정보 보호를 위해 PCA로 변환된 거래 특성 (의미 비공개) |
| Amount | 거래 금액(유로) |
| **Class** | **1 = 사기, 0 = 정상 (예측 대상)** |

### 데이터 특징
- 사기 거래는 전체의 약 **0.5%** 로 극단적으로 불균형합니다.
- 출처: Credit Card Fraud Detection (ULB Machine Learning Group, ODbL)을 대회용으로 재구성. 정상 거래는 일부만 샘플링했습니다.

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem1/`)에 **`submission.csv`** 로 저장

## 평가
**Average Precision (PR 곡선 아래 면적)** — 높을수록 좋습니다.

- 제출한 값의 **순서**로 계산하는 지표입니다. 사기일수록 큰 값을 주면 됩니다.
- 0/1 라벨보다 **0~1 사이 확률(점수)** 을 제출하는 것이 유리합니다.

## 초기 코드 (베이스라인)
아래 베이스라인은 로지스틱 회귀로 0/1 라벨을 예측해 제출합니다. 동작은 하지만 점수가 낮습니다.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv"); test = pd.read_csv(f"{DATA_DIR}/test.csv")
feats = [c for c in train.columns if c not in ("id", "Class")]
clf = LogisticRegression(max_iter=200).fit(train[feats], train.Class)
pd.DataFrame({"id": test.id, "Class": clf.predict(test[feats])}).to_csv("submission.csv", index=False)

## 제출 형식
`submission.csv`

| id | Class |
|---|---|
| 70000 | 0.0012 |
| 70001 | 0.9731 |

- 컬럼명 `id`, `Class`, 값은 실수(사기 확률 또는 점수)
- 포맷이 틀리면 0점 처리됩니다.

In [ ]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()